In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from datetime import datetime, timedelta

In [ ]:
# ─────────────────────────────────────────────
# Pesos por tipo de ocorrência
#
# Define a gravidade de cada categoria de crime.
# Esses pesos são usados na Célula 4 para calcular
# o peso_bruto de cada ocorrência no modelo de risco.
#
# Escala: 1.0 (baixo risco) → 5.0 (alto risco)
# Tipos não mapeados recebem peso neutro (1.0) via fillna
# ─────────────────────────────────────────────

mapa_peso_tipo = {
    "Ação policial":                 2.0,   # presença policial = risco moderado
    "Operação policial":             2.5,   # operações tendem a ser mais intensas
    "Tentativa/Roubo":               2.0,
    "Disputa":                       2.0,
    "Homicidio/Tentativa":           5.0,   # maior gravidade → maior peso
    "Briga":                         1.5,
    "Tentativa/Roubo de cargas":     2.0,
    "Ataque a civis":                3.0,
    "Tiros a esmo":                  1.8,
    "Arrastão":                      1.8,
    "Tentativa/Roubo a banco":       2.5,
    "Sequestro/Cárcere Privado":     3.5,
    "Disparo Acidental":             1.2,
    "Tortura":                       4.0,
    "Suicídio":                      1.0,
    "Outro":                         1.0,
    "Não identificado":              1.0,   # sem informação → peso neutro
}

In [ ]:
# ─────────────────────────────────────────────
# Carregamento dos dados reais
#
# Lê o CSV do Fogo Cruzado localizado na pasta
# data/ do projeto.
# N armazena o total de ocorrências para uso
# nos cálculos de percentual nos gráficos.
# ─────────────────────────────────────────────

df = pd.read_csv(r"../data\cross_fire.csv")
N = len(df)   # total de ocorrências no dataset

df.head()     # visualização rápida das primeiras linhas

In [ ]:
# ─────────────────────────────────────────────
# Cálculo dos pesos
#
# Replica as transformações do weights_estimator.py:
#
# 1. peso_tipo     → gravidade da ocorrência (mapa_peso_tipo)
#
# 2. peso_recencia → decaimento exponencial pelo tempo:
#       peso = e^(-λ * dias / 365.25)
#       λ = ln(2) / meia_vida
#       com meia_vida = 1 ano:
#         ocorrência de hoje   → peso 1.0
#         ocorrência de 1 ano  → peso 0.5
#         ocorrência de 2 anos → peso 0.25
#
# 3. peso_bruto    → peso_tipo × peso_recencia
# ─────────────────────────────────────────────

hoje = datetime.now()   # referência temporal para o cálculo de recência

# mapeia o tipo de crime ao seu peso de gravidade
# fillna(1.0) garante peso neutro para tipos não mapeados
df["peso_tipo"] = df["main_reason"].map(mapa_peso_tipo).fillna(1.0)

# parsing da coluna 'data' (formato: dd/mm/yyyy, HH:MM:SS)
t_atual = pd.Timestamp(hoje).normalize()              # data de hoje sem hora
dt = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
df["data_timestamp"] = dt.dt.normalize()              # data sem hora
df["hora"] = dt.dt.hour                               # hora extraída para o histograma

# decaimento exponencial com meia-vida de 1 ano
meia_vida = 1.0                                       # em anos
lambda_ = np.log(2) / meia_vida                       # constante de decaimento
df["dias_atras"] = (t_atual - df["data_timestamp"]).dt.days
df["peso_recencia"] = np.exp(-lambda_ * df["dias_atras"] / 365.25)

# peso final: combinação de gravidade e recência
df["peso_bruto"] = df["peso_recencia"] * df["peso_tipo"]

# visualização das colunas calculadas
df[["main_reason", "hora", "dias_atras", "peso_tipo", "peso_recencia", "peso_bruto"]].head(10)

In [ ]:
# ─────────────────────────────────────────────
# Configuração do tema visual
#
# Define paleta de cores e estilo dark para
# todos os gráficos. Centralizado aqui para
# facilitar ajustes futuros sem precisar
# alterar cada gráfico individualmente.
# ─────────────────────────────────────────────

DARK_BG    = "#0f1117"   # fundo escuro da figura
PANEL_BG   = "#1a1d27"   # fundo dos painéis/eixos
ACCENT     = "#e05c3a"   # laranja — picos e destaques principais
ACCENT2    = "#4ecdc4"   # verde-água — destaques secundários
ACCENT3    = "#f7c59f"   # bege — demais categorias
TEXT_MAIN  = "#f0f0f0"   # cor principal de texto
TEXT_SUB   = "#9aa3b2"   # cor secundária (labels, anotações)
GRID_COLOR = "#2a2d3a"   # cor da grade de fundo

# aplica o tema globalmente a todos os gráficos seguintes
plt.rcParams.update({
    "font.family":      "monospace",
    "text.color":       TEXT_MAIN,
    "axes.labelcolor":  TEXT_MAIN,
    "xtick.color":      TEXT_SUB,
    "ytick.color":      TEXT_SUB,
    "axes.facecolor":   PANEL_BG,
    "figure.facecolor": DARK_BG,
    "axes.edgecolor":   GRID_COLOR,
    "axes.grid":        True,
    "grid.color":       GRID_COLOR,
    "grid.linewidth":   0.6,
    "grid.linestyle":   "--",
})

In [ ]:
# ─────────────────────────────────────────────
# Gráfico 1: Distribuição por hora
#
# Mostra em quais horas do dia as ocorrências
# são mais frequentes.
#
# Coloração:
#   laranja cheio → pico noturno (20–23h)
#   verde-água    → pico matinal (06–09h)
#   laranja suave → demais horas
# ─────────────────────────────────────────────

fig, ax1 = plt.subplots(figsize=(18, 5))
fig.patch.set_facecolor(DARK_BG)

# contagem de ocorrências agrupadas por hora
contagem_hora = df["hora"].value_counts().sort_index()

bars = ax1.bar(contagem_hora.index, contagem_hora.values,
               color=ACCENT, alpha=0.85, width=0.75, zorder=3)

# coloração diferenciada por faixa horária
for bar in bars:
    h = int(bar.get_x() + bar.get_width() / 2)
    if h in range(20, 24):          # pico noturno
        bar.set_color(ACCENT);  bar.set_alpha(1.0)
    elif h in range(6, 10):         # pico matinal
        bar.set_color(ACCENT2); bar.set_alpha(0.95)
    else:                           # demais horas
        bar.set_color(ACCENT);  bar.set_alpha(0.40)

ax1.set_title("Distribuição por Hora do Dia", fontsize=11, color=TEXT_MAIN, pad=10, loc="left")
ax1.set_xlabel("Hora", fontsize=9, color=TEXT_SUB)
ax1.set_ylabel("Nº de Ocorrências", fontsize=9, color=TEXT_SUB)
ax1.set_xticks(range(24))
ax1.set_xticklabels([f"{h:02d}h" for h in range(24)], fontsize=7.5)
ax1.yaxis.set_major_locator(MaxNLocator(integer=True))

# legenda manual das faixas coloridas
ax1.annotate("■ Pico noturno (20–23h)", xy=(0.73, 0.88),
             xycoords="axes fraction", color=ACCENT, fontsize=8)
ax1.annotate("■ Pico matinal (06–09h)", xy=(0.73, 0.78),
             xycoords="axes fraction", color=ACCENT2, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Gráfico 2: Tipo de crime
#
# Barras horizontais com contagem por categoria
# de ocorrência (coluna main_reason).
# Ordenado do mais frequente para o menos.
#
# Coloração por gravidade:
#   laranja → Homicídio (peso 5.0)
#   verde   → Roubo e Tráfico (peso alto)
#   bege    → demais categorias
#
# Percentual exibido ao lado de cada barra
# para facilitar comparação proporcional.
# ─────────────────────────────────────────────

fig, ax2 = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor(DARK_BG)

# ordenar do maior para o menor volume
contagem_tipo = df["main_reason"].value_counts()

# cor baseada na gravidade do tipo de crime
cores_tipo = [ACCENT if t == "Homicídio"
              else ACCENT2 if t in ("Roubo", "Tráfico")
              else ACCENT3
              for t in contagem_tipo.index]

ax2.barh(contagem_tipo.index, contagem_tipo.values,
         color=cores_tipo, alpha=0.88, height=0.65, zorder=3)

# percentual ao lado de cada barra
for i, val in enumerate(contagem_tipo.values):
    ax2.text(val + 8, i, f"{val/N*100:.1f}%",
             va="center", fontsize=7.5, color=TEXT_SUB)

ax2.set_title("Tipo de Crime (main_reason)", fontsize=11, color=TEXT_MAIN, pad=10, loc="left")
ax2.set_xlabel("Nº de Ocorrências", fontsize=9, color=TEXT_SUB)
ax2.invert_yaxis()                                     # maior volume no topo
ax2.set_xlim(0, contagem_tipo.values[0] * 1.18)        # espaço para exibir o percentual

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Gráfico 3: Recência das ocorrências
#
# Histograma mostrando a distribuição dos dias
# desde cada ocorrência até hoje.
#
# A linha tracejada marca a meia-vida (365 dias):
#   → ocorrências à esquerda: peso_recencia > 0.5
#   → ocorrências à direita:  peso_recencia < 0.5
#
# O gradiente de opacidade nas barras reforça
# visualmente a concentração dos dados:
# barras mais altas ficam mais opacas.
# ─────────────────────────────────────────────

fig, ax3 = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor(DARK_BG)

# histograma com 30 faixas de dias
counts, edges, patches = ax3.hist(
    df["dias_atras"],
    bins=30,
    color=ACCENT2,
    alpha=0.80,
    edgecolor=DARK_BG,
    linewidth=0.4,
    zorder=3,
)

# gradiente de opacidade: barras mais altas ficam mais saturadas
max_count = counts.max()
for patch, count in zip(patches, counts):
    patch.set_alpha(0.35 + 0.65 * count / max_count)

# linha vertical marcando a meia-vida (onde peso_recencia = 0.5)
ax3.axvline(365, color=ACCENT, linewidth=1.5, linestyle="--", zorder=4)
ax3.text(370, max_count * 0.92, "meia-vida\n(1 ano)",
         color=ACCENT, fontsize=7.5, va="top")

ax3.set_title("Recência das Ocorrências", fontsize=11, color=TEXT_MAIN, pad=10, loc="left")
ax3.set_xlabel("Dias desde a ocorrência", fontsize=9, color=TEXT_SUB)
ax3.set_ylabel("Nº de Ocorrências", fontsize=9, color=TEXT_SUB)
ax3.yaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.show()